# Containerising the Campaign Analysis with Docker

In this practice you will package the skin clinic campaign analysis into a Docker image, so that it runs the same way on any machine without installing Python or libraries, and then share the image as a single file.

Everything here uses only what Lesson 5 covers: writing a Dockerfile with FROM, WORKDIR, COPY, RUN and CMD, building an image, running a container, and saving and loading images.

## Background

A skin clinic ran a marketing campaign targeting 10,000 customers. The clinic wants to know which customer segments are more likely to respond, based on demographics and purchase behaviour. The insights will help it target future campaigns and increase campaign effectiveness. The analysis works on your laptop; the operations team needs to run it on their own machines with no Python set-up.

**Before you start:** install **Docker Desktop** and check that it is running with `docker --version` in a terminal. Sections 1 and 2 run in this notebook; Sections 4 to 6 run in a terminal.

## Data Description

This practice uses **skin clinic campaign.csv** (10,000 customers, one row per customer), included in the download.

| Column Name | Description |
|---|---|
| CustID | Unique customer identifier |
| Gender | Female or Male |
| AgeGroup | <30, 30-50 or >50 |
| Purchase_Last_Quarter | Whether the customer made a purchase in the last quarter (Yes/No) |
| Unique_Products_Purchased | Number of different products bought in the last year (1 to 15) |
| Response_to_Campaign | **Target variable:** whether the customer responded to the campaign (Yes/No) |

## 1. Write the Analysis Script

The container will run a plain Python script, **app.py**, that loads the data and prints the response rate (%) by gender, age group, purchase in the last quarter, and product usage (1-4, 5-8, more than 8 products). We first create a project folder, **campaign-analysis**, and copy the dataset into it. Everything Docker needs goes in this folder, separate from the lesson's own example files.

In [1]:
from pathlib import Path
import shutil

project = Path("campaign-analysis")
project.mkdir(exist_ok=True)
shutil.copy("skin clinic campaign.csv", project)
print("Project folder ready:", sorted(f.name for f in project.iterdir()))

Project folder ready: ['skin clinic campaign.csv']


In [2]:
%%writefile campaign-analysis/app.py
import pandas as pd

# Load the data
df = pd.read_csv("skin clinic campaign.csv")

# Convert the response to 1 (Yes) and 0 (No)
df["Response"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

# Group the number of unique products purchased
def categorize_products(x):
    if x <= 4:
        return "1-4"
    elif x <= 8:
        return "5-8"
    else:
        return ">8"

df["Product_Category"] = df["Unique_Products_Purchased"].apply(categorize_products)

# Response rate (%) for each level of a column
def response_table(group_col):
    result = df.groupby(group_col)["Response"].mean().reset_index()
    result["Response Rate (%)"] = (result["Response"] * 100).round(2)
    return result.drop(columns=["Response"])


# Print the four tables when the container runs
for title, column in [("Gender", "Gender"),
                      ("Age Group", "AgeGroup"),
                      ("Purchase in Last Quarter", "Purchase_Last_Quarter"),
                      ("Product Usage", "Product_Category")]:
    print(f"\n{title} vs Campaign Response")
    print(response_table(column).to_string(index=False))

Writing campaign-analysis/app.py


## 2. Check the Script Before Containerising

Always confirm a script works before packaging it. Running it from the project folder, as the container will, prints the four tables the container should print.

In [3]:
import subprocess

result = subprocess.run(["python", "app.py"], cwd="campaign-analysis",
                        capture_output=True, text=True)
print(result.stdout or result.stderr)


Gender vs Campaign Response
Gender  Response Rate (%)
Female              43.77
  Male              34.08

Age Group vs Campaign Response
AgeGroup  Response Rate (%)
   30-50              47.06
     <30              32.93
     >50              32.60

Purchase in Last Quarter vs Campaign Response
Purchase_Last_Quarter  Response Rate (%)
                   No              21.69
                  Yes              49.65

Product Usage vs Campaign Response
Product_Category  Response Rate (%)
             1-4              18.08
             5-8              38.48
              >8              52.00



**Interpretation:** the overall response rate is about 40%. Customers aged 30-50, those who bought in the last quarter, and those who buy more than 8 different products respond best (around 47-52%). Customers who bought 1-4 products respond least (about 18%).

## 3. List the Dependencies and Write the Dockerfile

**requirements.txt** lists the libraries the script needs. The **Dockerfile** (no file extension) uses the five instructions from the lesson:

- **FROM python:3.10** sets a base image with Python already installed.
- **WORKDIR /app** sets the working directory inside the container.
- **COPY . .** copies the script, requirements and data into the container.
- **RUN pip install -r requirements.txt** installs pandas while the image is built.
- **CMD ["python", "app.py"]** runs the analysis when the container starts.

In [4]:
%%writefile campaign-analysis/requirements.txt
pandas

Writing campaign-analysis/requirements.txt


In [5]:
%%writefile campaign-analysis/Dockerfile
FROM python:3.10

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

CMD ["python", "app.py"]

Writing campaign-analysis/Dockerfile


The project folder now holds everything the image needs:

```text
campaign-analysis/
├── app.py
├── requirements.txt
├── Dockerfile
└── skin clinic campaign.csv
```

Docker copies the whole folder with `COPY . .`, so keep only these four files in it.

## 4. Build the Image

Open a terminal in the **campaign-analysis** folder and build an image called **campaign-analysis**. The `.` at the end tells Docker to use the Dockerfile in the current folder.

```bash
docker build -t campaign-analysis .
```

Check that the image exists with `docker images`.

## 5. Run the Container

```bash
docker run campaign-analysis
```

The container prints the same four tables as Section 2, with the same results on every machine.

## 6. Share the Image

Export the image to a single **.tar** file, then load it back and run it, as a colleague receiving the file would.

```bash
# Sender
docker save -o campaign-analysis.tar campaign-analysis

# Receiver
docker load -i campaign-analysis.tar
docker run campaign-analysis
```

The receiver needs only Docker: no Python, no pandas and no data set-up.

## Summary

- A Dockerfile packages code, libraries and data into an image using FROM, WORKDIR, COPY, RUN and CMD.
- `docker build` creates the image; `docker run` starts a container that runs the analysis.
- The container gives the same results on any machine, removing "it works on my laptop" problems.
- `docker save` and `docker load` share the image as a single file with colleagues who have only Docker installed.